# Lab 2 — two dialects behind one interface

*Day 1 · after Module 2*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadYusif/llm-application-engineering/blob/main/labs/lab2-two-providers.ipynb)

*Runs in Colab with no API key and nothing installed locally. The first cell fetches the course and starts the gateway, a small local service that answers from rules rather than from a model — so every number below is real about this harness, and not a claim about any provider.*

Module 2 covered the two wire dialects behind one interface, tokens counted per
route, the error taxonomy that decides whether to retry, and an open-weight model
served with no change above the boundary. Each of those is below, running against
**Murshid**.

The course gateway speaks both dialects, so you can watch the translation happen
without a key to either provider.

## Setup

In [1]:
import os, pathlib, re, subprocess, sys, time, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + r"\[[0-9;]*m")

REPO = "https://github.com/MohammadYusif/llm-application-engineering"
IN_COLAB = "google.colab" in sys.modules

# On Colab there is no checkout and no gateway, so fetch one and start one. The
# gateway is a local FastAPI app that answers from rules — no API key, no network
# calls out — which is the whole reason this course runs anywhere.
if IN_COLAB:
    root = pathlib.Path("/content/llm-application-engineering")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(root)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                        str(root / "murshid" / "requirements.lock")], check=True)
    os.chdir(root / "murshid")
else:
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (cand / "src" / "murshid").is_dir():
            os.chdir(cand); break
        if (cand / "murshid" / "src" / "murshid").is_dir():
            os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

# The application logs every routing decision and every model call. That is the
# point in production and noise in a notebook, so the default here is WARNING and
# the few sections where the log IS the lesson turn it back up themselves.
os.environ.setdefault("MURSHID_LOG_LEVEL", "WARNING")

def logs(level="INFO"):
    """Set the application's log level for the cells that follow."""
    os.environ["MURSHID_LOG_LEVEL"] = level
    from murshid.observability import configure_logging
    configure_logging()

def run(*args, quiet_logs=True, may_fail=False):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.

    may_fail=True for the commands whose job is to exit non-zero: the gate when
    it blocks, and the uncalibrated judge. Everywhere else a non-zero exit stops
    the notebook, because a traceback printed into a page that still reports as
    executed is worse than no output at all.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        # Structured logs come in two shapes — the console format on a laptop and
        # JSON lines in the container — so drop both, rather than whichever one
        # the machine that built this notebook happened to emit.
        def _is_log(line):
            if line.startswith("20") and "[" in line[:40]:
                return True
            return line.lstrip().startswith('{"') and (
                '"stage"' in line or '"event"' in line or '"logger"' in line)
        text = "\n".join(l for l in text.splitlines() if not _is_log(l))
    else:
        # The log is the lesson here, but not all of it: assistant_built and the
        # per-call llm_cost records are plumbing, and they are also the widest
        # lines on the page. Keep the retries, the failover and the refusals.
        NOISE = ("llm_cost", "assistant_built")
        text = "\n".join(l for l in text.splitlines()
                          if not any(n in l for n in NOISE))
    print(text.strip())
    if out.returncode and not may_fail:
        # A failing subprocess does not fail the notebook on its own, so say so
        # loudly. Without this a broken command is a traceback in the middle of a
        # page that still reports as executed cleanly.
        raise SystemExit(f"command failed with exit code {out.returncode}: {' '.join(args)}")
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

# demo_v0.py is deliberately naive — hardcoded model, inline prompt, no timeout —
# but it does read OPENAI_BASE_URL, and its default is only right on a laptop.
# Point it at the same gateway as everything else so the lab works in both places.
os.environ.setdefault("OPENAI_BASE_URL", GATEWAY + "/v1")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

def gateway_models(timeout=3):
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=timeout) as r:
        return json.load(r)["models"]

try:
    print("gateway:", gateway_models())
except Exception:
    if IN_COLAB:
        # Nothing is listening yet on a fresh runtime, so start it here. It runs
        # for the life of the notebook and needs no credentials.
        subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app",
                          "--host", "127.0.0.1", "--port", "8080", "--log-level", "warning"],
                         cwd="infra/mockgw",
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(60):
            try:
                print("gateway:", gateway_models(timeout=2)); break
            except Exception:
                time.sleep(1)
        else:
            print("the course gateway did not come up — re-run this cell")
    else:
        print(f"gateway at {GATEWAY} is NOT answering — start it first:")
        print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


## 1. Two wire dialects

The same conversation goes over the wire two different ways. OpenAI-compatible
APIs take a flat `messages` list with the system prompt as the first message.
Anthropic takes `system` as a **separate top-level field** and content as typed
blocks — so the adapter has to split them.

In [2]:
from murshid.llm.anthropic_client import split_system, to_anthropic_tools
from murshid.llm.interfaces import Message

conversation = [Message(role="system", content="You are Murshid, a services assistant."),
                Message(role="user", content="How much is a licence renewal?")]

system, body = split_system(conversation)
print("openai   : messages =", [m.role for m in conversation])
print("anthropic: system   =", repr(system))
print("           messages =", body)

openai   : messages = ['system', 'user']
anthropic: system   = 'You are Murshid, a services assistant.'
           messages = [{'role': 'user', 'content': [{'type': 'text', 'text': 'How much is a licence renewal?'}]}]


Tools are declared differently too. OpenAI wraps each one in `{"type": "function",
"function": {...}}` with `parameters`; Anthropic takes the tool flat with
`input_schema`. Same JSON Schema, two envelopes.

In [3]:
from murshid.tools.registry import tool_schemas

openai_tool = tool_schemas(["check_application_status"])[0]
anthropic_tool = to_anthropic_tools([openai_tool])[0]

print("openai   :", list(openai_tool), "->", list(openai_tool["function"]))
print("anthropic:", list(anthropic_tool))
print()
print("the schema itself is unchanged:",
      openai_tool["function"]["parameters"] == anthropic_tool["input_schema"])

openai   : ['type', 'function'] -> ['name', 'description', 'parameters']
anthropic: ['name', 'description', 'input_schema']

the schema itself is unchanged: True


Above the boundary none of that is visible. The same `LLMRequest` goes to both
routes and comes back as the same `LLMResponse` — only `model_id` differs.

In [4]:
from murshid.app import build_client
from murshid.config import get_settings
from murshid.llm.interfaces import LLMRequest

from murshid.domain.directory import rendered_directory

settings = get_settings()

# Same grounded question as Lab 1: the directory goes in the system message, so an
# answer that quotes a fee is quoting one that exists.
directory = rendered_directory("en")
question = [Message(role="system", content="Answer only from this directory.\n" + directory),
            Message(role="user", content="How much does a commercial licence renewal cost?")]

for route in ("primary", "comparison"):
    client = build_client(settings, route)
    reply = client.complete(LLMRequest(messages=question, model_alias="murshid-flagship",
                                       max_tokens=260))
    print(f"{route:<11} dialect={settings.route(route).dialect:<10} "
          f"model_id={reply.model_id:<17} finish={reply.finish_reason}")
    print(f"            {reply.text.strip().splitlines()[1][:88]}")

primary     dialect=openai     model_id=course-flagship   finish=stop
            - Fee: SAR 200 for each year of renewal
comparison  dialect=anthropic  model_id=course-anthropic  finish=stop
            - Fee: SAR 200 for each year of renewal


Two dialects, one call site. That is the whole argument for the adapter, and the
architecture test from Lab 1 is what stops the provider SDK leaking past it.

## 2. Sampling, determinism, and tokens

Temperature is a sampling parameter, not a truth dial. Even at `temperature=0` a
real provider can vary between calls — batching, hardware and model updates all
move the result — so an application that needs stability gets it from **structure**
(schemas, validators, tests) rather than from a hyperparameter.

In [5]:
client = build_client(settings, settings.primary_route)
answers = [client.complete(LLMRequest(messages=question, model_alias="murshid-flagship",
                                      temperature=0.0, max_tokens=120)).text
           for _ in range(3)]

print("three calls at temperature 0 identical?", len(set(answers)) == 1)
print()
print("On this gateway they are, because it answers from rules. Against a real")
print("provider, do not assume it — the module explains why, and Module 5 is how")
print("you would find out.")

three calls at temperature 0 identical? True

On this gateway they are, because it answers from rules. Against a real
provider, do not assume it — the module explains why, and Module 5 is how
you would find out.


Tokens are not words, and the count depends on the tokenizer the route uses. The
same two sentences, counted under two encodings, is where the Arabic premium
becomes visible.

In [6]:
from murshid.llm import tokens

en = "How much does it cost to renew a commercial registration?"
ar = "كم تبلغ رسوم تجديد السجل التجاري؟"

print(f"{'encoding':<14}{'english':>9}{'arabic':>9}   ratio")
for encoding in ("o200k_base", "cl100k_base"):
    e, a = tokens.count_with(en, encoding), tokens.count_with(ar, encoding)
    print(f"{encoding:<14}{e:>9}{a:>9}   {a / e:.2f}x")

print()
print("route encodings:",
      {m: tokens.encoding_for_model(m) for m in ("course-flagship", "murshid-onprem")})

encoding        english   arabic   ratio
o200k_base           11        9   0.82x


cl100k_base          11       24   2.18x

route encodings: {'course-flagship': 'o200k_base', 'murshid-onprem': 'cl100k_base'}


Under `cl100k_base` the Arabic sentence costs more than twice its English
counterpart; under `o200k_base` it does not. Same text, same meaning — the bill
depends on which route it went to. Count per route, on your own corpus, before
quoting anyone a price.

## 3. Streaming, rate limits, and the error taxonomy

Streaming buys perceived latency. The number to report is time-to-first-token,
alongside the total — one without the other hides the trade.

In [7]:
import time

start = time.perf_counter()
ttft, chunks, text = None, 0, []
for chunk in client.stream(LLMRequest(messages=question, model_alias="murshid-flagship",
                                      max_tokens=200)):
    if chunk.delta:
        if ttft is None:
            ttft = (time.perf_counter() - start) * 1000
        text.append(chunk.delta)
    chunks += 1
total = (time.perf_counter() - start) * 1000

print(f"chunks {chunks} | first token {ttft:.0f} ms | complete {total:.0f} ms")
print("".join(text).strip().splitlines()[1][:88])

chunks 99 | first token 62 ms | complete 130 ms
- Fee: SAR 200 for each year of renewal


Now a real rate limit. The gateway has a fault injector, so the 429 is served over
the wire rather than faked in Python — the adapter's error mapping runs for real.

In [8]:
from murshid.llm.interfaces import LLMError

print(fault({"mode": "rate_limit", "seconds": 20, "retry_after": 2}))

try:
    client.complete(LLMRequest(messages=question, model_alias="murshid-flagship", max_tokens=60))
except LLMError as exc:
    print()
    print("class     :", type(exc).__name__)
    print("status    :", exc.status)
    print("retryable :", exc.retryable)
    print("retry_after:", exc.retry_after, "seconds — the server said when, so honour it")

{'fault': {'mode': 'rate_limit', 'until': 1788705851.320034, 'retry_after': 2}}

class     : LLMError
status    : 429
retryable : True
retry_after: 2.0 seconds — the server said when, so honour it


`retryable` is the whole taxonomy in one flag. 429 and 5xx are retryable with
backoff; 400, 401 and a context-length error are not, and retrying them just burns
quota and time. Anything the adapter does not recognise is **not** retryable —
guessing wrong in that direction is cheaper.

The resilient client reads that flag and nothing else.

In [9]:
from murshid.llm.resilient import ResilientClient

resilient = ResilientClient([("primary", client)], max_attempts=3, sleep=lambda _: None)
try:
    resilient.complete(LLMRequest(messages=question, model_alias="murshid-flagship",
                                  max_tokens=60))
except Exception as exc:
    print(type(exc).__name__, "-> every hop exhausted while the fault is on")

print(fault({"mode": "off"}), "\n")
reply = resilient.complete(LLMRequest(messages=question, model_alias="murshid-flagship",
                                      max_tokens=60))
print("fault cleared, same client:", reply.model_id, "|", reply.text.strip()[:70])

2026-09-06T14:43:51.331292Z [warning  ] llm_retry                      attempt=1 delay_s=2.0 hop=primary retry_after=2.0 status=429


2026-09-06T14:43:51.333797Z [warning  ] llm_retry                      attempt=2 delay_s=2.0 hop=primary retry_after=2.0 status=429


2026-09-06T14:43:51.329454Z [warning  ] llm_hop_exhausted              attempts=3 hop=primary


AllHopsExhausted -> every hop exhausted while the fault is on
{'fault': {'mode': 'off'}} 



fault cleared, same client: course-flagship | About Renewing a commercial registration (CR):
- Fee: SAR 200 for each


## 4. Serving open-weight models

The `vllm` route is an OpenAI-compatible server the application talks to with the
same adapter — the only difference above the boundary is which route name it asks
for, and that its residency is `on_premise`.

In [10]:
for name in ("primary", "vllm"):
    route = settings.route(name)
    print(f"{name:<9} dialect={route.dialect:<8} residency={route.residency:<12} "
          f"model={route.resolve('murshid-flagship')}")

onprem = build_client(settings, "vllm")
reply = onprem.complete(LLMRequest(messages=question, model_alias="murshid-flagship",
                                   max_tokens=120))
print()
print("answered by:", reply.model_id, "| route:", reply.route)
print(reply.text.strip().splitlines()[1][:88])

primary   dialect=openai   residency=cloud        model=course-flagship
vllm      dialect=openai   residency=on_premise   model=murshid-onprem



answered by: murshid-onprem | route: vllm
- Fee: SAR 200 for each year of renewal


No code above the boundary changed — the route name did. That is the claim the
capstone rubric scores, and the reason it is worth the adapter: an on-premise
option stays open, and data residency becomes a configuration decision instead of
a rewrite.

## 5. Commercial versus open-weight, decided like an engineer

Not "which is better" — *which one, for which slice of traffic, at what cost and
what latency*. The cost meter turns that into a table you can defend.

In [11]:
from murshid.observability.cost import CostMeter

meter = CostMeter(settings.prices)
rows = []
for name in ("primary", "cheap", "comparison", "vllm"):
    c = build_client(settings, name)
    started = time.perf_counter()
    r = c.complete(LLMRequest(messages=question, model_alias="murshid-flagship", max_tokens=120))
    ms = (time.perf_counter() - started) * 1000
    record = meter.meter(r, route=name, intent="faq")
    rows.append((name, r.model_id, ms, r.usage.input_tokens, r.usage.output_tokens,
                 record.cost_halalas))

print(f"{'route':<11}{'model':<18}{'ms':>7}{'in':>6}{'out':>6}{'halalas':>10}")
for name, model, ms, tin, tout, cost in rows:
    print(f"{name:<11}{model:<18}{ms:>7.0f}{tin:>6}{tout:>6}{cost:>10.4f}")

cheapest = min(rows, key=lambda r: r[5])
dearest = max(rows, key=lambda r: r[5])
print()
print(f"{dearest[0]} costs {dearest[5] / cheapest[5]:.0f}x {cheapest[0]} on this one question")

route      model                  ms    in   out   halalas
primary    course-flagship        97  1150   120    0.8211
cheap      course-small           46  1150   120    0.0347
comparison course-anthropic      121  1146   163    1.0585
vllm       murshid-onprem        107  1150   120    0.2311

comparison costs 31x cheap on this one question


One question is not a benchmark. What makes this a decision rather than an anecdote
is running it over a corpus, at realistic concurrency, and reporting p50 **and**
p95 — a mean latency hides exactly the tail your users complain about.

The caveat that has to travel with every number here: this is the course gateway,
answering from rules. The **shape** of the comparison is real; the magnitudes are
this harness's, not any provider's.

In [12]:
print("total metered this lab:", round(meter.total_halalas, 4), "halalas")
print("by route:", {k: round(v, 4) for k, v in meter.by("route").items()})

total metered this lab: 2.1455 halalas
by route: {'comparison': 1.0585, 'primary': 0.8211, 'vllm': 0.2311, 'cheap': 0.0347}


## 6. Common mistakes

- **Assuming `temperature=0` means reproducible.** It does not; structure gives you
  stability, sampling parameters do not.
- **Counting tokens with one tokenizer for every route.** The Arabic premium above
  is the counter-example, and it changes the bill.
- **Retrying everything, or nothing.** The taxonomy is the point: retry 429 and
  5xx with backoff and jitter, fail fast on 400 and 401.
- **Comparing providers on one question.** A corpus, at concurrency, with p50 and
  p95 — otherwise it is a demo.
- **Treating open-weight as free.** It is a different cost shape — GPUs you rent by
  the hour rather than tokens you buy — and Module 6 works out the break-even.

## Your turn — on your own project

The same two backends, on your own application:

1. **Two live routes, switchable by config** — one commercial, one open-weight. The
   claim the rubric scores is that swapping them is an environment variable, so
   prove it the way Lab 1's contract suite does: one test class, every adapter.
2. **Map the error taxonomy once**, in your adapter, into a single retryable flag.
   Anything unrecognised is not retryable.
3. **Run your own fault drill** and keep the log excerpt. A fallback chain that has
   never been exercised scores nothing.
4. **Start your `BENCHMARKS.md`** with a provider table you produced: p50 and p95,
   cost per call, and token counts under each route's own tokenizer.

**Next:** [Module 3 — structured outputs and tools](../modules/m3-structured-outputs-and-tools.qmd),
then [Lab 3](lab3-tickets-and-tools.ipynb).